In [2]:
import os
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from PIL import Image
import numpy as np
from pathlib import Path
import math

In [5]:
def calculate_grid_size(n):
    """Calculate the optimal grid size for n images using predefined layouts."""
    # 预定义常见数量的最优布局
    layouts = {
        # 1: (1, 1),
        # 2: (1, 2),
        # 3: (2, 2),
        # 4: (2, 2),
        # 5: (2, 3),
        # 6: (2, 3),
        # 7: (2, 4),
        # 8: (2, 4),
        # 9: (3, 3),
        # 10: (3, 4),
        # 11: (3, 4),
        # 12: (3, 4),
        # 可以继续添加更多...
    }
    
    # 如果数量在预定义布局中，直接返回
    if n in layouts:
        layout = layouts[n]
        print(f"Calculated grid size for {n} images: {layout}")
        return layout
    
    # 对于未预定义的数量，使用一个简单的计算方法
    rows = int(math.sqrt(n))
    cols = math.ceil(n / rows)
    layout = (rows, cols)
    # print(f"Calculated grid size for {n} images: {layout}")
    return layout

def load_image(path):
    """Load and convert image to numpy array."""
    img = Image.open(path)
    img = img.convert('RGB')
    return np.array(img)

def plot_comparison(folders_to_plot, image_name, compact_mode=False, figsize=(20, 12), dpi=300):
    """Plot comparison of same image from different folders."""
    n = len(folders_to_plot)
    rows, cols = calculate_grid_size(n)
    
    # 首先读取第一张图片来获取尺寸比例
    first_img = load_image(os.path.join(next(iter(folders_to_plot.values())), image_name))
    aspect_ratio = first_img.shape[1] / first_img.shape[0]  # 宽/高
    
    if compact_mode:
        # 根据图像比例调整figsize
        if figsize[0]/figsize[1] > (cols*aspect_ratio)/(rows):
            # 以高度为基准
            new_height = figsize[1]
            new_width = new_height * (cols*aspect_ratio)/rows
        else:
            # 以宽度为基准
            new_width = figsize[0]
            new_height = new_width * rows/(cols*aspect_ratio)
        
        fig = plt.figure(figsize=(new_width, new_height), frameon=False, dpi=dpi)
        gs = gridspec.GridSpec(rows, cols)
        gs.update(wspace=0, hspace=0, left=0, right=1, bottom=0, top=1)
    else:
        base_size = 5
        figsize = (base_size * cols, base_size * rows)
        fig = plt.figure(figsize=figsize, facecolor='white', dpi=dpi)
        gs = gridspec.GridSpec(rows, cols)
        gs.update(wspace=0.1, hspace=0.2)
    
    for idx, (folder_name, folder_path) in enumerate(folders_to_plot.items()):
        if idx < n:
            ax = plt.subplot(gs[idx])
            img_path = os.path.join(folder_path, image_name)
            img = load_image(img_path)
            
            # 使用 'equal' 而不是 'auto' 来保持原始比例
            ax.imshow(img, aspect='equal')
            ax.axis('off')
            
            if not compact_mode:
                ax.set_title(folder_name, pad=10)
            
            if compact_mode:
                ax.set_position([
                    ax.get_position().x0,
                    ax.get_position().y0,
                    ax.get_position().width,
                    ax.get_position().height
                ])
    
    if not compact_mode:
        plt.tight_layout()
    
    return fig

def plot_all_comparisons(folders_to_plot, save_dir=None, compact_mode=False):
    """Plot comparisons for all images."""
    original_folder = folders_to_plot['original']
    image_files = [f for f in os.listdir(original_folder) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    
    total_images = len(image_files)
    for idx, img_name in enumerate(image_files, 1):
        print(f"Processing image {idx}/{total_images}: {img_name}")
        fig = plot_comparison(folders_to_plot, img_name, compact_mode=compact_mode)
        
        if save_dir:
            mode_suffix = '_compact' if compact_mode else '_annotated'
            save_path = os.path.join(save_dir, f'comparison{mode_suffix}_{img_name}')
            Path(save_dir).mkdir(parents=True, exist_ok=True)
            
            if compact_mode:
                # 紧凑模式：完全无边距保存
                fig.savefig(save_path, 
                          bbox_inches='tight',
                          pad_inches=0,
                          facecolor='none',
                          transparent=True)
            else:
                fig.savefig(save_path, 
                          bbox_inches='tight',
                          facecolor='white')
            plt.close(fig)
        else:
            plt.show()

In [6]:
base_path = "/root/Lecter/cyclegan-exp/us-hand-to-large"

folders_to_plot = {
    # 'original': os.path.join(base_path, 'datasets/all/test'),
    # 'resnet_9block': os.path.join(base_path, 'results/test_1.resnet_9block_x4_3090'),
    # # 'unet256_resize': os.path.join(base_path, 'results/test_2.raw_unet256_resize_256_3090'),
    # 'unet_256': os.path.join(base_path, 'results/test_2.unet_256_x256_3090'),
    # # 'hybrid_restormer': os.path.join(base_path, 'results/test_xxx.hybrid_restormer_1'),
    # 'vq_resnet_max256': os.path.join(base_path, 'results/test_3.vq_resnet_x4_max256_3090'),
    # 'vq_resnet_max512': os.path.join(base_path, 'results/test_3.vq_resnet_x4_max512_3090'),
    # 'vq_resnet_max1024': os.path.join(base_path, 'results/test_3.vq_resnet_x4_max1024_act1000_3090'),
    # 'vq_resnet_patch256_overlap32': os.path.join(base_path, 'results/test_patch256_overlap32'),
    # 'vq_resnet_patch256_overlap64': os.path.join(base_path, 'results/test_patch256_overlap64'),
    # 'vq_resnet_patch256_overlap128': os.path.join(base_path, 'results/test_patch256_overlap128'),
    # 'vq_resnet_patch512_overlap64': os.path.join(base_path, 'results/test_patch512_overlap64'),
    # 'vq_resnet_patch512_overlap128': os.path.join(base_path, 'results/test_patch512_overlap128'),
    # 'vq_resnet_patch512_overlap256': os.path.join(base_path, 'results/test_patch512_overlap256'),
    'original': '/root/Lecter/dcm-convert/degrade/t1090000101al-dir_raw',
    'degrade more(input)': '/root/Lecter/dcm-convert/degrade/t1090000101al-dir_degrade3',
    # 'resnet': '/root/Lecter/cyclegan-exp/us-hand-to-large/results/test-resnet-t1090000101al-dir_degrade_sr',
    # 'hyper_1': '/root/Lecter/1.pytorch-CycleGAN-and-pix2pix_latest/results/t1090000101al-dir_degrade_14.hyper_degrader3_clear1',
    # 'hyper_2': '/root/Lecter/1.pytorch-CycleGAN-and-pix2pix_latest/results/t1090000101al-dir_degrade_14.hyper_degrader3_clear2',
    'sr(output)': '/root/Lecter/1.pytorch-CycleGAN-and-pix2pix_latest/results/t1090000101al-dir_degrade3_14.hyper_degrader3_clear3',
    # 'denosise_100%': '/root/Lecter/dcm-convert/degrade/jiang-denoised/t1090000101al-dir_degrade_denoisedresults1',
    # 'denosise_50%': '/root/Lecter/dcm-convert/degrade/jiang-denoised/t1090000101al-dir_degrade_denoisedresults2',
    # 'denosise_20%': '/root/Lecter/dcm-convert/degrade/jiang-denoised/t1090000101al-dir_degrade_denoisedresults3',
    # t5070000101al-dir_compair_resnet

}

# 使用示例
# 定义保存目录（可选）
# save_dir = os.path.join(base_path, 'results/comparisons/comparisons_i-res-unet-resize-patch256-512_rotate/compact')
save_dir = '/root/Lecter/dcm-convert/degrade/t1090000101al-dir_compair_degrade3_hyper3'

# 绘制所有比较图
plot_all_comparisons(folders_to_plot, save_dir, compact_mode=False)

# # 如果只想查看单张图片的比较
# image_name = "20240328092259___.jpg"  # 替换为实际的图片名称
# plot_comparison(folders_to_plot, image_name, compact_mode=True, dpi=600)
# plt.show()

Processing image 1/534: t1090000101al_frame0123.png


/tmp/ipykernel_3447315/178339887.py:91: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Processing image 2/534: t1090000101al_frame0141.png
Processing image 3/534: t1090000101al_frame0188.png
Processing image 4/534: t1090000101al_frame0224.png
Processing image 5/534: t1090000101al_frame0166.png
Processing image 6/534: t1090000101al_frame0133.png
Processing image 7/534: t1090000101al_frame0487.png
Processing image 8/534: t1090000101al_frame0226.png
Processing image 9/534: t1090000101al_frame0117.png
Processing image 10/534: t1090000101al_frame0367.png
Processing image 11/534: t1090000101al_frame0492.png
Processing image 12/534: t1090000101al_frame0009.png
Processing image 13/534: t1090000101al_frame0423.png
Processing image 14/534: t1090000101al_frame0046.png
Processing image 15/534: t1090000101al_frame0303.png
Processing image 16/534: t1090000101al_frame0437.png
Processing image 17/534: t1090000101al_frame0355.png
Processing image 18/534: t1090000101al_frame0049.png
Processing image 19/534: t1090000101al_frame0002.png
Processing image 20/534: t1090000101al_frame0167.png
P